In [ ]:
import os
import time
from pymmcore_plus import CMMCorePlus
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QScrollArea, QTabWidget, QFileDialog,
    QRadioButton, QFrame, QSpinBox, QLineEdit, QCheckBox, QPushButton,
    QLabel, QHBoxLayout, QVBoxLayout, QComboBox, QWidget, QTableWidget,
    QTableWidgetItem, QMessageBox, QInputDialog, QGridLayout, QSizePolicy,
    QGroupBox, QSplitter
)
from PyQt5.QtGui import QPixmap, QImage, QFont
from PyQt5.QtCore import QTimer, pyqtSignal, pyqtSlot, Qt, QThread, QSize
from PyQt5 import QtGui, QtCore, QtWidgets
import serial
import numpy as np
import pandas as pd
import sys
from PIL import Image
import csv
import cv2
from skimage import exposure
from collections import defaultdict
from matplotlib.figure import Figure
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
import copy
import tifffile as tiff


# ─────────────────────────────────────────────
#  Hardware constants
# ─────────────────────────────────────────────
Arduino_port      = "COM4"
Arduino_baud_rate = 115200
Arduino_timeout   = 0.1

Relay_port    = 'COM6'
Relay_baudrate = 19200
Relay_timeout  = 0.1


# ─────────────────────────────────────────────
#  Style helpers
# ─────────────────────────────────────────────
BTN_RED    = "background-color: #bb283a; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_GREEN  = "background-color: #2ac555; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_BLUE   = "background-color: #3a6bc9; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_YELLOW = "background-color: #d4a017; color: black; border-radius: 4px; padding: 4px 8px;"
BTN_BLACK  = "background-color: #222222; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_PURPLE = "background-color: #6a3db8; color: white; border-radius: 4px; padding: 4px 8px;"

LABEL_BOLD = "font-weight: bold; color: #cccccc;"
GROUP_STYLE = """
QGroupBox {
    border: 1px solid #555555;
    border-radius: 6px;
    margin-top: 10px;
    padding-top: 4px;
    color: #aaaaaa;
    font-weight: bold;
}
QGroupBox::title {
    subcontrol-origin: margin;
    left: 8px;
    padding: 0 4px;
}
"""

WIDGET_STYLE = """
    QTableWidget, QTableView {
        background-color: #1e1e1e;
        color: #ffffff;
        gridline-color: #444444;
        border: 1px solid #444444;
    }
    QTableWidget::item { color: #ffffff; background-color: #1e1e1e; }
    QTableWidget::item:selected { background-color: #2a82da; color: white; }
    QHeaderView::section {
        background-color: #2d2d2d;
        color: #cccccc;
        border: 1px solid #444444;
        padding: 3px;
    }
    QComboBox {
        background-color: #2d2d2d;
        color: #ffffff;
        border: 1px solid #555555;
        border-radius: 4px;
        padding: 3px 6px;
    }
    QComboBox QAbstractItemView {
        background-color: #2d2d2d;
        color: #ffffff;
        selection-background-color: #2a82da;
    }
    QComboBox::drop-down { border: none; }
    QSpinBox, QDoubleSpinBox {
        background-color: #2d2d2d;
        color: #ffffff;
        border: 1px solid #555555;
        border-radius: 4px;
        padding: 3px 6px;
    }
    QSpinBox::up-button, QSpinBox::down-button,
    QDoubleSpinBox::up-button, QDoubleSpinBox::down-button {
        background-color: #3a3a3a;
        border: none;
    }
    QLineEdit {
        background-color: #2d2d2d;
        color: #ffffff;
        border: 1px solid #555555;
        border-radius: 4px;
        padding: 3px 6px;
    }
    QScrollBar:vertical, QScrollBar:horizontal {
        background-color: #1e1e1e;
        border: none;
    }
    QScrollBar::handle:vertical, QScrollBar::handle:horizontal {
        background-color: #555555;
        border-radius: 3px;
        min-height: 20px;
    }
    QCheckBox { color: #cccccc; }
    QLabel { color: #cccccc; }
"""


DARK_PALETTE = {
    "Window":          (53,  53,  53),
    "WindowText":      (255, 255, 255),
    "Base":            (25,  25,  25),
    "AlternateBase":   (53,  53,  53),
    "ToolTipBase":     (255, 255, 255),
    "ToolTipText":     (255, 255, 255),
    "Text":            (255, 255, 255),
    "Button":          (53,  53,  53),
    "ButtonText":      (255, 255, 255),
    "BrightText":      (255, 0,   0),
    "Link":            (42,  130, 218),
    "Highlight":       (42,  130, 218),
    "HighlightedText": (0,   0,   0),
}


def make_dark_palette():
    p = QtGui.QPalette()
    mapping = {
        "Window":          QtGui.QPalette.Window,
        "WindowText":      QtGui.QPalette.WindowText,
        "Base":            QtGui.QPalette.Base,
        "AlternateBase":   QtGui.QPalette.AlternateBase,
        "ToolTipBase":     QtGui.QPalette.ToolTipBase,
        "ToolTipText":     QtGui.QPalette.ToolTipText,
        "Text":            QtGui.QPalette.Text,
        "Button":          QtGui.QPalette.Button,
        "ButtonText":      QtGui.QPalette.ButtonText,
        "BrightText":      QtGui.QPalette.BrightText,
        "Link":            QtGui.QPalette.Link,
        "Highlight":       QtGui.QPalette.Highlight,
        "HighlightedText": QtGui.QPalette.HighlightedText,
    }
    for name, role in mapping.items():
        p.setColor(role, QtGui.QColor(*DARK_PALETTE[name]))
    return p


def group(title, layout, flat=False):
    """Wrap a layout in a styled QGroupBox."""
    box = QGroupBox(title)
    box.setStyleSheet(GROUP_STYLE)
    box.setFlat(flat)
    box.setLayout(layout)
    return box


def hline():
    line = QFrame()
    line.setFrameShape(QFrame.HLine)
    line.setFrameShadow(QFrame.Sunken)
    line.setStyleSheet("color: #444444;")
    return line


def labeled_input(label_text, widget, label_width=None):
    """Return an HBoxLayout with a label + widget."""
    lbl = QLabel(label_text)
    lbl.setStyleSheet("color: #aaaaaa;")
    if label_width:
        lbl.setFixedWidth(label_width)
    row = QHBoxLayout()
    row.addWidget(lbl)
    row.addWidget(widget)
    return row


# ─────────────────────────────────────────────
#  Hardware init
# ─────────────────────────────────────────────
def initialize_arduino(port, baud_rate, timeout):
    try:
        arduino = serial.Serial(port, baud_rate, timeout=timeout)
        time.sleep(2)
        print(f"Connected to {port} at {baud_rate} baud.")
        return arduino
    except serial.SerialException:
        print(f"Error: Could not open serial port {port}.")
        return None

def handshake(arduino):
    if arduino:
        for _ in range(5):
            arduino.write(b"HELLO\n")
            time.sleep(0.5)
            response = arduino.readline().decode('utf-8').strip()
            if response == "READY":
                print("Handshake successful!")
                return True
        print("Handshake failed!")
        return False
    return False

arduino = initialize_arduino(Arduino_port, Arduino_baud_rate, Arduino_timeout)
if arduino and handshake(arduino):
    print("Serial connection established.")
else:
    print("Failed to establish a connection. Exiting.")
    if arduino:
        arduino.close()
    exit()

def send_to_arduino(data):
    arduino.write(f"{data}\n".encode())
    time.sleep(0.1)
    response = arduino.readline().decode().strip()
    print(f"Arduino Response: {response}" if response else "No response received.")

def set_voltage(set_value):
    if str(set_value) == 'High':
        send_to_arduino(5)
    elif str(set_value) == 'Low':
        send_to_arduino(4)

    print(f"Voltage set to: {str(set_value)}")


def init_serial_port(relayport, relay_Baudrate, relay_timeout):
    try:
        obj = serial.Serial(relayport, relay_Baudrate, timeout=relay_timeout)
        obj.write(b'')
        obj.flush()
        obj.write_terminator = b'\r'
        print("Numato relay correctly connected")
        return obj
    except serial.SerialException:
        print("Numato relay NOT correctly connected")
        sys.exit(1)

Numato_device = init_serial_port(Relay_port, Relay_baudrate, Relay_timeout)

Microscope = {}
Microscope['mmc'] = CMMCorePlus.instance()
Microscope['mmc'].loadSystemConfiguration("C:\\MATLAB Microscope\\AmoghMMConfig_Hamamatsu.cfg")


# ─────────────────────────────────────────────
#  Custom toggle button
# ─────────────────────────────────────────────
class QToggleButton(QPushButton):
    def __init__(self, text='', parent=None):
        super().__init__(text, parent)
        self.setCheckable(True)
        self.setChecked(False)
        self.setStyleSheet(BTN_RED)


# ─────────────────────────────────────────────
#  Video thread
# ─────────────────────────────────────────────
class VideoThread(QThread):
    change_pixmap_signal = pyqtSignal(np.ndarray)

    def __init__(self):
        super().__init__()
        self._run_flag    = True
        self._record_flag = False
        self._lock        = QtCore.QMutex()          # FIX: guards self.out across threads
        self.out          = None
        self.video_directory = "C:/Users/Cell Culture Scope/Downloads/Videos"
        self.video_filename  = self.get_unique_filename(self.video_directory)
        self.fourcc      = cv2.VideoWriter_fourcc(*'MJPG')
        self.fps         = 10
        self.frame_size  = None
        self.mmc         = Microscope['mmc']
        self.camera      = self.mmc.getCameraDevice()
        self.DIAshutter  = 'TIDiaShutter'
        self.focus       = self.mmc.getFocusDevice()
        self.stage       = self.mmc.getXYStageDevice()
        self.PFS         = self.mmc.getAutoFocusDevice()
        self.EPIshutter  = 'TIEpiShutter'
        self.DIAlamp     = 'TIDiaLamp'
        self.scope       = 'TIScope'
        self.zoom        = 'TINosePiece'
        self.filter      = 'TIFilterBlock1'
        self.lightpath   = 'TILightPath'
        self.PFS_offset  = 'TIPFSOffset'
        self.core        = 'Core'
        self.camerapath  = '2-Left100'
        self.mmc.setProperty(self.camera, "CONVERSION FACTOR COEFF", "0.5")

    def run(self):
        self.mmc.setProperty(self.DIAshutter, 'State', 0)
        self.mmc.setProperty(self.EPIshutter,  'State', 0)
        self.mmc.setProperty(self.core, 'Shutter', self.DIAshutter)
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        self.mmc.initializeCircularBuffer()
        self.mmc.prepareSequenceAcquisition(self.camera)
        self.mmc.waitForDevice(self.DIAshutter)
        self.mmc.waitForDevice(self.camera)
        self.mmc.startContinuousSequenceAcquisition(100)
        while self._run_flag:
            # FIX: poll isSequenceRunning() every iteration instead of caching
            # a one-shot bool; catches camera timeout / buffer-overrun recovery
            if not self.mmc.isSequenceRunning():
                print("VideoThread: sequence stopped unexpectedly, exiting run loop")
                break
            if self.mmc.getRemainingImageCount() > 0:
                try:
                    liveimage    = self.mmc.getLastImage()
                    image_width  = self.mmc.getImageWidth()
                    image_height = self.mmc.getImageHeight()
                except Exception as e:
                    print(f"VideoThread: frame grab error – {e}")
                    continue
                self.final_image = self.convert_raw_np(liveimage, image_width, image_height, np.uint16)
                self.change_pixmap_signal.emit(self.final_image)
                # FIX: lock around all access to self.out so stop_recording()
                # called from the main thread cannot release the writer while
                # we are mid-write here in the worker thread
                self._lock.lock()
                try:
                    if self._record_flag:
                        if self.out is None:
                            self.video_filename = self.get_unique_filename(self.video_directory)
                            # FIX: derive frame_size from actual array shape, not
                            # from mmc width/height which can be stale after a
                            # circular-buffer overrun
                            h, w = self.final_image.shape[:2]
                            self.frame_size = (w, h)
                            self.out = cv2.VideoWriter(
                                self.video_filename, self.fourcc, self.fps,
                                self.frame_size, isColor=True)
                            if not self.out.isOpened():
                                print(f"VideoThread: failed to open writer at {self.video_filename}")
                                self.out = None
                        if self.out is not None and self.out.isOpened():
                            frame_rescaled = exposure.rescale_intensity(
                                self.final_image, in_range='image', out_range='uint8').astype(np.uint8)
                            frame_to_save = cv2.cvtColor(frame_rescaled, cv2.COLOR_GRAY2BGR)
                            # FIX: validate frame dimensions before writing to
                            # avoid VideoWriter crash on unexpected buffer size
                            if frame_to_save.shape[:2][::-1] == self.frame_size:
                                try:
                                    self.out.write(frame_to_save)
                                except Exception as e:
                                    print(f"VideoThread: write error – {e}")
                                    self._record_flag = False
                                    self.out.release(); self.out = None
                            else:
                                print(f"VideoThread: frame size mismatch "
                                      f"{frame_to_save.shape[:2][::-1]} vs {self.frame_size}, skipping")
                finally:
                    self._lock.unlock()
            else:
                QtCore.QThread.msleep(5)   # avoid busy-spin when buffer is empty
        self.mmc.stopSequenceAcquisition(self.camera)
        self.mmc.clearCircularBuffer()
        self._lock.lock()
        try:
            if self.out:
                self.out.release(); self.out = None
        finally:
            self._lock.unlock()

    def convert_raw_np(self, raw_img, img_width, img_height, pixel_Type):
        rawImage = np.frombuffer(raw_img, dtype=pixel_Type).reshape((img_height, img_width)).T
        return exposure.rescale_intensity(rawImage)

    def start_recording(self):
        # FIX: set flag under lock so the run loop sees a consistent state
        self._lock.lock()
        self._record_flag = True
        self._lock.unlock()

    def stop_recording(self):
        # FIX: acquire lock before touching self.out to prevent concurrent release
        self._lock.lock()
        try:
            self._record_flag = False
            if self.out:
                self.out.release(); self.out = None
        finally:
            self._lock.unlock()

    def get_unique_filename(self, directory, base="output", ext=".avi"):
        os.makedirs(directory, exist_ok=True)
        i, filename = 1, os.path.join(directory, f"{base}{ext}")
        while os.path.exists(filename):
            filename = os.path.join(directory, f"{base}_{i}{ext}"); i += 1
        return filename

    def stop(self):
        self._lock.lock()
        self._record_flag = False
        if self.out:
            self.out.release(); self.out = None
        self._lock.unlock()
        self._run_flag = False
        self.wait()


# ─────────────────────────────────────────────
#  Droplet worker thread
# ─────────────────────────────────────────────
class DropletWorker(QThread):
    def __init__(self, operation, input_1, input_2=None,
                 purge_duration=0, flow_duration=0, drive_duration=0,
                 chemostat_number=4, PWM_duration1=0.05, PWM_duration2=0.05,
                 PWM_totalduration=5, connect=False, chemostat_number2=None):
        super().__init__()
        self.Numato_port       = Numato_device
        self.operation         = operation
        self.input_1           = input_1
        self.input_2           = input_2
        self.chemostat         = chemostat_number
        self.Purge_duration    = purge_duration
        self.Flow_duration     = flow_duration
        self.Drive_duration    = drive_duration
        self.PWM_duration1     = PWM_duration1
        self.PWM_duration2     = PWM_duration2
        self.PWM_totalduration = PWM_totalduration
        self.connect           = connect
        self.chemostat_number2 = chemostat_number2

    def run(self):
        ops = {
            "purge":      lambda: self.purge_inlet(self.input_1, self.input_2),
            "generate":   lambda: self.generate_droplet(self.input_1, self.input_2),
            "drive":      lambda: self.drive_droplet(self.chemostat, self.connect, self.chemostat_number2),
            "characterize": lambda: self.characterize_droplet(self.chemostat),
            "wash":       lambda: self.wash_step(self.input_1, self.input_2),
        
        }

        ops.get(self.operation, lambda: print("Invalid operation"))()

    # ── valve helpers ──────────────────────────────────────
    def send_relay_command(self, command):
        if self.Numato_port and self.Numato_port.is_open:
            try:
                self.Numato_port.write(f"{command}\r".encode('utf-8'))
                time.sleep(0.005)
            except serial.SerialException:
                print('Worker thread failed to communicate with device')

    def get_relay_id(self, idx):
        return str(idx) if idx <= 9 else chr(ord('A') + (idx - 10))

    def control_valve(self, idx, state):
        relay_id = self.get_relay_id(idx)
        self.send_relay_command(f"relay {'off' if state else 'on'} {relay_id}")

    # ── operations ────────────────────────────────────────
    def characterize_droplet(self, chemostat_number):
        for v in [8, 9, 10, 11]: self.control_valve(v, state=False)
        time.sleep(3)
        self.control_valve(15, state=False); self.control_valve(7, state=False)
        time.sleep(self.Purge_duration)
        self.control_valve(2, state=True);  self.control_valve(5, state=True)
        time.sleep(0.5)
        self.control_valve(7, state=True);  self.control_valve(4, state=False)
        self.control_valve(3, state=False)
        time.sleep(self.Flow_duration)
        self.control_valve(3, state=True);  self.control_valve(4, state=True)
        time.sleep(2)
        self.control_valve(15, state=True)
        self.control_valve(2, state=False); self.control_valve(5, state=False)
        self.control_valve(chemostat_number + 7, state=True)
        send_to_arduino(2)
        time.sleep(self.Drive_duration)
        self.control_valve(chemostat_number + 7, state=False)
        send_to_arduino(3)
        self.control_valve(4, state=False)
        time.sleep(1)
        self.control_valve(4, state=True)

    def purge_inlet(self, input_1, input_2):
        self.control_valve(input_1 + 11, state=False)
        time.sleep(1)
        self.control_valve(7, state=False)
        self.control_valve(input_2 + 11, state=False)
        time.sleep(self.Purge_duration)
        self.control_valve(input_1 + 11, state=True)
        self.control_valve(input_2 + 11, state=True)
        self.control_valve(7, state=True)
        time.sleep(20)

    def generate_droplet(self, input_1, input_2):
        self.control_valve(2, state=True);  self.control_valve(5, state=True)
        self.control_valve(15, state=False); time.sleep(0.1)
        self.control_valve(3, state=False); self.control_valve(4, state=False)
        time.sleep(self.Flow_duration)
        self.control_valve(3, state=True);  self.control_valve(4, state=True)
        self.control_valve(15, state=True)

    def drive_droplet(self, Chemostat_number, connect=False, chemostat_number2=None):
        """Drive droplet for Chemostat_number.
        If connect=True, also open chemostat_number2's valve simultaneously,
        effectively connecting the two chemostats during the drive step.
        """
        self.control_valve(2, state=False); self.control_valve(5, state=False)
        self.control_valve(Chemostat_number + 7, state=True)
        if connect and chemostat_number2 is not None:
            # Open the target chemostat valve as well to connect them
            self.control_valve(chemostat_number2 + 7, state=True)
        send_to_arduino(2)
        time.sleep(self.Drive_duration)
        send_to_arduino(3)
        self.control_valve(Chemostat_number + 7, state=False)
        if connect and chemostat_number2 is not None:
            self.control_valve(chemostat_number2 + 7, state=False)
        self.control_valve(4, state=False)
        time.sleep(3)
        self.control_valve(4, state=True)

    def wash_step(self, input_1, input_2):
        self.control_valve(input_1 + 11, state=False)
        self.control_valve(input_2 + 11, state=False)
        time.sleep(4)
        self.control_valve(input_2 + 11, state=True)
        self.control_valve(7, state=False)
        time.sleep(5)
        self.control_valve(input_1 + 11, state=True)
        self.control_valve(7, state=True)



# ═══════════════════════════════════════════════════════════
#  MAIN GUI
# ═══════════════════════════════════════════════════════════
class MicroscopeControlGUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.Numato_port  = Numato_device
        self.video_thread = None
        self.positions    = []
        self.selected_exposures = []
        self.Chemostat_protocol_steps = []
        self.current_protocol_table_step = 0
        self._init_microscope()
        self._build_ui()
        self.setWindowTitle('Microscope Control')
        self._aspect_w = 1160
        self._aspect_h = 1000
        self.resize(self._aspect_w, self._aspect_h)
        self.show()

    # ── microscope init ───────────────────────────────────
    def _init_microscope(self):
        self.mmc = Microscope['mmc']
        self.camera     = self.mmc.getCameraDevice()
        self.DIAshutter = self.mmc.getShutterDevice()
        self.focus      = self.mmc.getFocusDevice()
        self.stage      = self.mmc.getXYStageDevice()
        self.PFS        = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        self.DIAlamp    = 'TIDiaLamp'
        self.scope      = 'TIScope'
        self.zoom       = 'TINosePiece'
        self.filter     = 'TIFilterBlock1'
        self.lightpath  = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core       = 'Core'
        self.eyepath    = '1-Eye100'
        self.camerapath = '2-Left100'
        self.zoom4x  = '1-(Achromat) 4x NA 0.10 Dry'
        self.zoom10x = '2-(Achromat) 10x NA 0.25 Dry'
        self.zoom20x = '3-(Achromat) 20x NA 0.40 Dry'
        self.zoom40x = '4-S Plan Fluor 40x NA 0.60 Dry'
        self.zoom60x = '5-Plan Apo 60x NA 1.40 Oil'
        self.zoomempty = '6-Unknown'
        self.filterNames = {1:'1- FITC', 2:'2- DAPI', 3:'3- BFP-A',
                            4:'4- Cy5',  5:'5- Cy3',  6:'6- DIA'}
        # camera setup
        self.mmc.setProperty(self.camera, 'Sensor Cooler', 'ON')
        self.mmc.setProperty(self.camera, 'Exposure', 20)
        self.mmc.setProperty(self.camera, 'MINIMUM ACQUISITION TIMEOUT', 500)
        self.mmc.setProperty(self.camera, 'Binning', '4x4')
        self.mmc.setProperty(self.camera, 'ScanMode', 1)
        self.mmc.setProperty(self.camera, 'CONVERSION FACTOR COEFF', 0.5)
        self.mmc.setProperty(self.camera, 'PixelType', '16bit')
        self.mmc.setProperty(self.DIAlamp, 'ComputerControl', 'On')
        self.mmc.setProperty(self.DIAlamp, 'Intensity', 4)
        self.mmc.setProperty(self.DIAlamp, 'State', 0)
        self.mmc.setProperty(self.DIAshutter, 'State', 0)
        self.mmc.setProperty(self.EPIshutter,  'State', 0)
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)
        os.chdir('C:/Users/Cell Culture Scope/Documents/MATLAB')

    # ══════════════════════════════════════════
    #  UI builder
    # ══════════════════════════════════════════
    def _build_ui(self):
        self.tabs = QTabWidget()
        self.tabs.setStyleSheet("""
            QTabWidget::pane { border: 1px solid #444; }
            QTabBar::tab { background: #333; color: #aaa; padding: 6px 18px; border-radius: 4px 4px 0 0; }
            QTabBar::tab:selected { background: #555; color: white; }
        """)
        self.setCentralWidget(self.tabs)

        tab1 = QWidget(); self.tabs.addTab(tab1, "Microscope Control")
        tab2 = QWidget(); self.tabs.addTab(tab2, "Timelapse Setup")

        self._build_tab1(tab1)
        self._build_tab2(tab2)

    # ══════════════════════════════════════════
    #  TAB 1 — Microscope Control
    # ══════════════════════════════════════════
    def _build_tab1(self, parent):
        root = QHBoxLayout(parent)
        root.setSpacing(10)
        root.setContentsMargins(10, 10, 10, 10)

        # ── LEFT PANEL ─────────────────────────
        left = QVBoxLayout()
        left.setSpacing(8)

        # -- Illumination group
        illum_grid = QGridLayout()
        illum_grid.setSpacing(6)
        self.dialamponlight = QToggleButton("DIA Lamp")
        self.dialamponlight.clicked.connect(self.DIAlamp_ON)
        self.DIAshutterbutton = QToggleButton("DIA Shutter")
        self.DIAshutterbutton.clicked.connect(self.toggle_DIA_shutter)
        self.EPIshutterbutton = QToggleButton("EPI Shutter")
        self.EPIshutterbutton.clicked.connect(self.toggle_EPI_shutter)
        illum_grid.addWidget(self.dialamponlight,    0, 0)
        illum_grid.addWidget(self.DIAshutterbutton,  0, 1)
        illum_grid.addWidget(self.EPIshutterbutton,  0, 2)
        left.addWidget(group("Illumination", illum_grid))

        # -- Objective + light path
        obj_layout = QHBoxLayout()
        obj_layout.setSpacing(6)
        self.Zoom_list = QComboBox()
        self.Zoom_list.addItems([self.zoom4x, self.zoom10x, self.zoom20x,
                                  self.zoom40x, self.zoom60x, self.zoomempty])
        self.Zoom_list.currentTextChanged.connect(self.Set_zoom)
        self.eyepathlight    = QPushButton("→ Eye")
        self.camerapathlight = QPushButton("→ Camera")
        for btn in (self.eyepathlight, self.camerapathlight):
            btn.setStyleSheet(BTN_BLUE)
            btn.clicked.connect(self.PathtoCamera)
        obj_layout.addWidget(self.Zoom_list, 2)
        obj_layout.addWidget(self.camerapathlight, 1)
        obj_layout.addWidget(self.eyepathlight,    1)
        left.addWidget(group("Objective & Light Path", obj_layout))

        # -- Filter buttons
        filter_grid = QGridLayout()
        filter_grid.setSpacing(4)
        for fk, fv in self.filterNames.items():
            btn = QPushButton(fv)
            btn.setStyleSheet(BTN_PURPLE)
            btn.clicked.connect(lambda checked, fn=fk: self.change_filter(fn))
            r, c = divmod(fk - 1, 3)
            filter_grid.addWidget(btn, r, c)
        left.addWidget(group("Filters", filter_grid))

        # -- Image viewer (fixed, never collapses)
        self.image_Live = QLabel()
        self.image_Live.setAlignment(Qt.AlignCenter)
        self.image_Live.setStyleSheet("background-color: #111111; border-radius: 4px;")
        self.image_Live.setMinimumSize(340, 260)
        self.image_Live.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.image_Live.setText("No image loaded")

        # -- Image capture buttons (always above the viewer)
        cap_layout = QHBoxLayout()
        cap_layout.setSpacing(6)
        self.snap_Button   = QPushButton("Snap Image")
        self.live_Button   = QToggleButton("Live Image")
        self.save_Button   = QPushButton("Save Image")
        self.record_button = QPushButton("Start Recording")
        self.record_button.setCheckable(True)
        for btn, style in [(self.snap_Button,   BTN_BLUE),
                           (self.live_Button,   BTN_RED),
                           (self.save_Button,   BTN_BLUE),
                           (self.record_button, BTN_RED)]:
            btn.setStyleSheet(style)
        self.snap_Button.clicked.connect(self.snap_DIA_image)
        self.live_Button.clicked.connect(self.startstoplive_imaging)
        self.save_Button.clicked.connect(self.save_image)
        self.record_button.clicked.connect(self.handle_record_button)
        for btn in (self.snap_Button, self.live_Button, self.save_Button, self.record_button):
            cap_layout.addWidget(btn)

        # -- Snap Fluorescent Image controls (below image viewer)
        self.quick_EPI_filter   = QSpinBox(); self.quick_EPI_filter.setRange(1,6); self.quick_EPI_filter.setValue(4)
        self.quick_EPI_exposure = QLineEdit("100")
        self.snap_EPI_button    = QPushButton("Snap Fluorescent Image")
        self.snap_EPI_button.setStyleSheet(BTN_BLUE)
        self.snap_EPI_button.clicked.connect(
            lambda: self.snap_EPI_image(self.quick_EPI_filter.value(), int(self.quick_EPI_exposure.text())))
        snap_epi_row = QHBoxLayout(); snap_epi_row.setSpacing(6)
        qef_lbl = QLabel("Filter:"); qef_lbl.setStyleSheet("color:#aaa;")
        qee_lbl = QLabel("Exposure (ms):"); qee_lbl.setStyleSheet("color:#aaa;")
        snap_epi_row.addWidget(qef_lbl)
        snap_epi_row.addWidget(self.quick_EPI_filter)
        snap_epi_row.addWidget(qee_lbl)
        snap_epi_row.addWidget(self.quick_EPI_exposure)
        snap_epi_row.addWidget(self.snap_EPI_button)

        viewer_layout = QVBoxLayout()
        viewer_layout.setSpacing(4)
        viewer_layout.addLayout(cap_layout)
        viewer_layout.addWidget(self.image_Live, 1)
        viewer_layout.addLayout(snap_epi_row)
        left.addWidget(group("Image Viewer", viewer_layout), 1)

        ''' # -- Stage controls
        stage_grid = QGridLayout()
        stage_grid.setSpacing(4)
        self.stagefast, self.stagemedium, self.stageslow = 1000, 100, 10
        self.stage_speed = self.stagefast
        self.stagespeedbutton = QComboBox()
        self.stagespeedbutton.addItems([str(self.stageslow), str(self.stagemedium), str(self.stagefast)])
        self.stagespeedbutton.setCurrentIndex(2)
        self.stagespeedbutton.currentTextChanged.connect(self.Set_stage_speed)
        self.Xplus  = QPushButton("X+"); self.Xminus = QPushButton("X−")
        self.Yplus  = QPushButton("Y+"); self.Yminus = QPushButton("Y−")
        self.Zplus  = QPushButton("Z+"); self.Zminus = QPushButton("Z−")
        for btn in (self.Xplus, self.Xminus, self.Yplus, self.Yminus, self.Zplus, self.Zminus):
            btn.setStyleSheet(BTN_BLUE)
        self.Xplus.clicked.connect( lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage)+self.stage_speed, self.mmc.getYPosition(self.stage)))
        self.Xminus.clicked.connect(lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage)-self.stage_speed, self.mmc.getYPosition(self.stage)))
        self.Yplus.clicked.connect( lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage), self.mmc.getYPosition(self.stage)+self.stage_speed))
        self.Yminus.clicked.connect(lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage), self.mmc.getYPosition(self.stage)-self.stage_speed))
        self.Zplus.clicked.connect( lambda: self.mmc.setPosition(self.mmc.getPosition()+self.stage_speed))
        self.Zminus.clicked.connect(lambda: self.mmc.setPosition(self.mmc.getPosition()-self.stage_speed))
        speed_lbl = QLabel("Step (µm):"); speed_lbl.setStyleSheet("color:#aaa;")
        stage_grid.addWidget(self.Xplus,  0, 0); stage_grid.addWidget(self.Xminus, 0, 1)
        stage_grid.addWidget(self.Yplus,  1, 0); stage_grid.addWidget(self.Yminus, 1, 1)
        stage_grid.addWidget(self.Zplus,  2, 0); stage_grid.addWidget(self.Zminus, 2, 1)
        stage_grid.addWidget(speed_lbl,   3, 0); stage_grid.addWidget(self.stagespeedbutton, 3, 1)
        left.addWidget(group("Stage", stage_grid)) '''

        # ── RIGHT PANEL ────────────────────────
        right = QVBoxLayout()
        right.setSpacing(8)

        # -- Saved positions
        pos_layout = QVBoxLayout()
        pos_layout.setSpacing(4)
        self.Positions_table = QTableWidget(self)
        self.Positions_table.setRowCount(8)
        self.Positions_table.setColumnCount(3)
        self.Positions_table.setHorizontalHeaderLabels(['X', 'Y', 'Z'])
        self.Positions_table.verticalHeader().setDefaultSectionSize(22)
        self.Positions_table.setMaximumHeight(220)
        #self.Positions_table.horizontalHeader().setStretchLastSection(True)
        pos_btns = QHBoxLayout()
        pos_btns.setSpacing(4)
        self.save_Position_button  = QPushButton("Add Position")
        self.replacePositionButton = QPushButton("Replace")
        self.clearButton           = QPushButton("Clear All")
        self.GoToPositionButton = QPushButton("Go to Position:")
        for btn, style in [(self.save_Position_button,  BTN_GREEN),
                           (self.replacePositionButton, BTN_BLUE),
                           (self.GoToPositionButton, BTN_BLACK),
                           (self.clearButton, BTN_RED)]:
            btn.setStyleSheet(style)
        
        self.save_Position_button.clicked.connect(self.save_Position)
        #self.replacePositionButton.clicked.connect(self.replacePosition)
        self.Positions_spinbox  = QSpinBox(); self.Positions_spinbox.setRange(1, 8); self.Positions_spinbox.setValue(1)
        self.replacePositionButton.clicked.connect(lambda: self.replacePosition(self.Positions_spinbox.value()))
        self.GoToPositionButton.clicked.connect(lambda: self.GoToPosition(self.Positions_spinbox.value()))

        self.clearButton.clicked.connect(self.clearPositions)
        for btn in (self.GoToPositionButton, self.Positions_spinbox, self.replacePositionButton, self.save_Position_button, self.clearButton):
            pos_btns.addWidget(btn)
        pos_layout.addLayout(pos_btns)
        pos_layout.addWidget(self.Positions_table)
        right.addWidget(group("Saved Positions (max 8)", pos_layout))

        # -- Valve grid
        valve_grid = QGridLayout()
        valve_grid.setSpacing(4)
        self.controls = []
        for i in range(21):
            btn = QPushButton(f"V{i+1}")
            btn.setCheckable(True)
            btn.setStyleSheet(BTN_RED)
            btn.setFixedHeight(28)
            btn.clicked.connect(lambda state, idx=i: self.control_valve(idx, state))
            self.controls.append(btn)
            valve_grid.addWidget(btn, i // 7, i % 7)
        valve_btns = QHBoxLayout()
        self.stop_all_button = QPushButton("Stop All")
        self.stop_all_button.setStyleSheet(BTN_BLACK)
        self.all_on_button   = QPushButton("All On")
        self.all_on_button.setStyleSheet(BTN_BLUE)
        self.stop_all_button.clicked.connect(self.stop_all_callback)
        self.all_on_button.clicked.connect(self.all_on_callback)
        valve_btns.addWidget(self.stop_all_button)
        valve_btns.addWidget(self.all_on_button)
        full_valve = QVBoxLayout()
        full_valve.setSpacing(4)
        full_valve.addLayout(valve_grid)
        full_valve.addLayout(valve_btns)
        right.addWidget(group("Valve Controls", full_valve))

        # -- Droplet parameters
        dp_grid = QGridLayout()
        dp_grid.setSpacing(4)
        self.purge_duration_Input  = QLineEdit("0")
        self.flow_duration_Input   = QLineEdit("0")
        self.drive_duration_Input  = QLineEdit("0")
        self.inlet_Input           = QLineEdit("1")
        for row, (lbl, widget) in enumerate([
            ("Purge duration (s):",       self.purge_duration_Input),
            ("Aqueous flow duration (s):", self.flow_duration_Input),
            ("Drive duration (s):",        self.drive_duration_Input),
            ("Chemostat number:",          self.inlet_Input),
        ]):
            l = QLabel(lbl); l.setStyleSheet("color:#aaa;")
            dp_grid.addWidget(l, row, 0)
            dp_grid.addWidget(widget, row, 1)
        self.Generate_drop_button = QPushButton("Generate Droplet")
        self.Generate_drop_button.setStyleSheet(BTN_GREEN)
        self.Generate_drop_button.clicked.connect(lambda: self.Characterize_Droplet(input=3))
        dp_grid.addWidget(self.Generate_drop_button, 4, 0, 1, 2)
        right.addWidget(group("Droplet Parameters", dp_grid))

        # -- Voltage
        volt_layout = QVBoxLayout(); volt_layout.setSpacing(6)
        volt_row = QHBoxLayout()
        v_lbl = QLabel("Operating voltage (V):"); v_lbl.setStyleSheet("color:#aaa;")
        self.voltagevalues = QComboBox()
        self.voltagevalues.addItems(['High', 'Low'])
        self.voltagevalues.currentTextChanged.connect(set_voltage)
        volt_row.addWidget(v_lbl); volt_row.addWidget(self.voltagevalues)
        self.TurnOnVolts = QPushButton("Voltage Signal")
        self.TurnOnVolts.setCheckable(True)
        self.TurnOnVolts.setStyleSheet(BTN_RED)
        self.TurnOnVolts.clicked.connect(self.alter_Arduino_state)
        volt_layout.addLayout(volt_row)
        volt_layout.addWidget(self.TurnOnVolts)
        right.addWidget(group("Voltage", volt_layout))
        
        right.addStretch()

        root.addLayout(left, 3)
        root.addLayout(right, 2)

    # ══════════════════════════════════════════
    #  TAB 2 — Timelapse Setup
    # ══════════════════════════════════════════
    def _build_tab2(self, parent):
        root = QVBoxLayout(parent)
        root.setSpacing(8)
        root.setContentsMargins(10, 10, 10, 10)

        # ══════════════════════════════════════════
        # ROW 1: Fluorescence Imaging | Timing | Save Directory | Experiment Control
        # ══════════════════════════════════════════
        row1 = QHBoxLayout()
        row1.setSpacing(8)

        # -- Fluorescence Imaging --
        fe_layout = QVBoxLayout()
        fe_layout.setSpacing(4)
        fe_top = QHBoxLayout()
        fe_top.setSpacing(8)

        filter_list_layout = QVBoxLayout()
        filter_list_layout.setSpacing(0)
        filter_list_layout.setContentsMargins(0, 24, 0, 0)
        for fk, fv in self.filterNames.items():
            lbl = QLabel(fv)
            lbl.setStyleSheet("color:#aaa; font-size:11px;")
            lbl.setFixedHeight(24)
            filter_list_layout.addWidget(lbl)
        filter_list_layout.addStretch()

        self.Exposures_table = QTableWidget(len(self.filterNames), 2)
        self.Exposures_table.setHorizontalHeaderLabels(["Filter (1-6)", "Exposure (ms)"])
        self.Exposures_table.verticalHeader().hide()
        self.Exposures_table.verticalHeader().setDefaultSectionSize(24)
        self.Exposures_table.setMaximumHeight(24 * len(self.filterNames) + 28)
        for row in range(len(self.filterNames)):
            sb1 = QSpinBox(); sb1.setRange(1, 6); sb1.setValue(row + 1)
            sb2 = QSpinBox(); sb2.setRange(0, 1000); sb2.setValue(0)
            self.Exposures_table.setCellWidget(row, 0, sb1)
            self.Exposures_table.setCellWidget(row, 1, sb2)
        self.Exposures_table.horizontalHeader().setStretchLastSection(False)
        self.Exposures_table.horizontalHeader().setSectionResizeMode(0, QtWidgets.QHeaderView.Fixed)
        self.Exposures_table.horizontalHeader().setSectionResizeMode(1, QtWidgets.QHeaderView.Fixed)
        self.Exposures_table.setColumnWidth(0, 90)
        self.Exposures_table.setColumnWidth(1, 100)
        self.Exposures_table.setFixedWidth(200)
        self.Exposures_table.setSizePolicy(QSizePolicy.Fixed, QSizePolicy.Fixed)

        fe_top.addLayout(filter_list_layout)
        fe_top.addWidget(self.Exposures_table)
        fe_top.addStretch()

        fe_bottom = QHBoxLayout()
        self.Save_Exposures_button = QPushButton("Save Values")
        self.Save_Exposures_button.setStyleSheet(BTN_GREEN)
        self.Save_Exposures_button.setFixedWidth(120)
        self.Save_Exposures_button.clicked.connect(self.read_Exposure_values)
        fe_bottom.addWidget(self.Save_Exposures_button)
        fe_bottom.addStretch()
        fe_layout.addLayout(fe_top)
        fe_layout.addLayout(fe_bottom)
        row1.addWidget(group("Fluorescence Imaging", fe_layout))

        # -- Timing --
        timing_grid = QGridLayout(); timing_grid.setSpacing(6)
        self.cycle_Interval_Input = QLineEdit("0")
        self.cycle_Input          = QLineEdit("0")
        for r, (lbl, w) in enumerate([
            ("Cycle interval (min):", self.cycle_Interval_Input),
            ("No. of cycles:",        self.cycle_Input)
        ]):
            l = QLabel(lbl); l.setStyleSheet("color:#aaa;")
            timing_grid.addWidget(l, r, 0)
            timing_grid.addWidget(w, r, 1)
        self.interval = 0; self.cycles = 0
        timing_wrap = QVBoxLayout()
        timing_wrap.addLayout(timing_grid)
        timing_wrap.addStretch()
        row1.addWidget(group("Timing", timing_wrap))

        # -- Save Directory --
        dir_layout = QVBoxLayout(); dir_layout.setSpacing(4)
        dir_row = QHBoxLayout(); dir_row.setSpacing(4)
        self.directory_input = QLineEdit()
        self.directory_input.setPlaceholderText("Save directory…")
        self.browse_button   = QPushButton("Browse")
        self.browse_button.setStyleSheet(BTN_BLUE)
        self.browse_button.clicked.connect(self.browse_folder)
        dir_row.addWidget(self.directory_input, 1)
        dir_row.addWidget(self.browse_button)
        dir_layout.addLayout(dir_row)
        dir_layout.addStretch()
        row1.addWidget(group("Save Directory", dir_layout))

        # -- Experiment Control --
        exp_layout = QVBoxLayout(); exp_layout.setSpacing(6)
        self.start_experiment_button = QPushButton("▶  Start Experiment")
        self.stop_experiment_button  = QPushButton("■  Stop Experiment")
        self.start_experiment_button.setStyleSheet(BTN_GREEN)
        self.stop_experiment_button.setStyleSheet(BTN_RED)
        self.start_experiment_button.setMinimumHeight(36)
        self.stop_experiment_button.setMinimumHeight(36)
        self.start_experiment_button.clicked.connect(
            lambda: self.TimeLapse_Experiment(
                num_loops=int(self.cycle_Input.text()),
                time_interval=int(self.cycle_Interval_Input.text()),
                positions_table=self.positions,
                selected_exposures=self.selected_exposures,
                chemostat_protocol_table=self.Chemostat_protocol_steps))
        exp_layout.addWidget(self.start_experiment_button)
        exp_layout.addWidget(self.stop_experiment_button)
        exp_layout.addStretch()
        row1.addWidget(group("Experiment Control", exp_layout))

        root.addLayout(row1)

        # ══════════════════════════════════════════
        # ROW 2: Protocol Definition
        # ══════════════════════════════════════════
        PROTO_FONT = "font-size: 13px;"
        PROTO_LABEL = f"color:#aaa; {PROTO_FONT}"
        PROTO_CB    = f"color:#ccc; {PROTO_FONT}"

        proto_top = QHBoxLayout()
        proto_top.setSpacing(16)
        proto_top.setContentsMargins(8, 8, 8, 8)

        # inputs spinboxes
        inp_layout = QGridLayout(); inp_layout.setSpacing(8)
        self.Chemostat_inputs = []
        for i in range(4):
            lbl = QLabel(f"Input {i+1}:"); lbl.setStyleSheet(PROTO_LABEL)
            sb  = QSpinBox(); sb.setRange(0, 5); sb.setValue(0)
            sb.setFixedWidth(60); sb.setStyleSheet(f"font-size: 13px;")
            inp_layout.addWidget(lbl, i, 0)
            inp_layout.addWidget(sb,  i, 1)
            self.Chemostat_inputs.append(sb)
        proto_top.addWidget(group("Inputs", inp_layout), 1)

        # chemostat checkboxes
        chem_layout = QVBoxLayout(); chem_layout.setSpacing(5)
        self.Chemostats = []
        for i in range(8):
            cb = QCheckBox(f"Chemostat {i+1}"); cb.setStyleSheet(PROTO_CB)
            chem_layout.addWidget(cb); self.Chemostats.append(cb)
        proto_top.addWidget(group("Chemostats", chem_layout), 1)

        # connections beside chemostats
        conn_outer = QVBoxLayout(); conn_outer.setSpacing(6)
        self.connection_rows = []
        for ci in range(4):
            row_layout = QHBoxLayout(); row_layout.setSpacing(6)
            enable_cb = QCheckBox("Chemostat")
            enable_cb.setStyleSheet(PROTO_CB)
            sb_a = QSpinBox(); sb_a.setRange(1, 8); sb_a.setValue(ci + 1)
            sb_a.setFixedWidth(50); sb_a.setStyleSheet("font-size: 13px;")
            lbl_with = QLabel("→"); lbl_with.setStyleSheet(PROTO_LABEL)
            sb_b = QSpinBox(); sb_b.setRange(1, 8); sb_b.setValue(min(ci + 2, 8))
            sb_b.setFixedWidth(50); sb_b.setStyleSheet("font-size: 13px;")
            row_layout.addWidget(enable_cb)
            row_layout.addWidget(sb_a)
            row_layout.addWidget(lbl_with)
            row_layout.addWidget(sb_b)
            row_layout.addStretch()
            conn_outer.addLayout(row_layout)
            self.connection_rows.append((enable_cb, sb_a, sb_b))
        freq_row = QHBoxLayout(); freq_row.setSpacing(6)
        freq_lbl = QLabel("Every"); freq_lbl.setStyleSheet(PROTO_LABEL)
        self.connection_every_n = QSpinBox()
        self.connection_every_n.setRange(1, 999); self.connection_every_n.setValue(1)
        self.connection_every_n.setFixedWidth(56); self.connection_every_n.setStyleSheet("font-size: 13px;")
        freq_lbl2 = QLabel("loop(s)"); freq_lbl2.setStyleSheet(PROTO_LABEL)
        freq_row.addWidget(freq_lbl)
        freq_row.addWidget(self.connection_every_n)
        freq_row.addWidget(freq_lbl2)
        freq_row.addStretch()
        conn_outer.addStretch()
        conn_outer.addLayout(freq_row)
        proto_top.addWidget(group("Connections", conn_outer), 2)

        # step buttons
        step_btn_layout = QVBoxLayout(); step_btn_layout.setSpacing(8)
        self.add_Step_button    = QPushButton("Add Step")
        self.clear_last_button  = QPushButton("Clear Last Step")
        self.clear_Steps_button = QPushButton("Clear All Steps")
        self.export_button      = QPushButton("Export to CSV")
        BTN_GREEN_LG  = BTN_GREEN  + " font-size: 13px; padding: 6px 10px;"
        BTN_BLUE_LG   = BTN_BLUE   + " font-size: 13px; padding: 6px 10px;"
        BTN_RED_LG    = BTN_RED    + " font-size: 13px; padding: 6px 10px;"
        BTN_BLACK_LG  = BTN_BLACK  + " font-size: 13px; padding: 6px 10px;"
        self.add_Step_button.setStyleSheet(BTN_GREEN_LG)
        self.clear_last_button.setStyleSheet(BTN_BLUE_LG)
        self.clear_Steps_button.setStyleSheet(BTN_RED_LG)
        self.export_button.setStyleSheet(BTN_BLACK_LG)
        self.add_Step_button.clicked.connect(self.add_loading_step)
        self.clear_last_button.clicked.connect(self.clear_last_step)
        self.clear_Steps_button.clicked.connect(self.clear_all_steps)
        self.export_button.clicked.connect(self.export_to_csv)
        for btn in (self.add_Step_button, self.clear_last_button, self.clear_Steps_button, self.export_button):
            step_btn_layout.addWidget(btn)
        step_btn_layout.addStretch()
        proto_top.addWidget(group("Actions", step_btn_layout), 1)

        root.addWidget(group("Protocol Definition", proto_top))

        # ══════════════════════════════════════════
        # ROW 3: Protocol Steps (fills remaining vertical space)
        # ══════════════════════════════════════════
        self.Chemostat_protocol_table = QTableWidget()
        self.Chemostat_protocol_table.setRowCount(10)
        self.Chemostat_protocol_table.setColumnCount(8)
        self.Chemostat_protocol_table.setVerticalHeaderLabels(
            ["Input 1", "Input 2"] + [f"Chemostat {i+1}" for i in range(8)])
        for c in range(8):
            self.Chemostat_protocol_table.setHorizontalHeaderItem(c, QTableWidgetItem(f"Step {c+1}"))
        for c in range(8):
            for r in range(10):
                if r < 2:
                    self.Chemostat_protocol_table.setItem(r, c, QTableWidgetItem(""))
                else:
                    self.Chemostat_protocol_table.setCellWidget(r, c, self.create_centered_checkbox())
        self.Chemostat_protocol_table.setStyleSheet("QTableWidget { gridline-color: #444; }")
        self.Chemostat_protocol_table.verticalHeader().setDefaultSectionSize(24)
        proto_tbl_layout = QVBoxLayout()
        proto_tbl_layout.addWidget(self.Chemostat_protocol_table)
        root.addWidget(group("Protocol Steps", proto_tbl_layout), 1)

    # ══════════════════════════════════════════
    #  Microscope slots
    # ══════════════════════════════════════════
    def startstoplive_imaging(self):
        if self.live_Button.isChecked():
            self.live_Button.setStyleSheet(BTN_GREEN)
            self.video_thread = VideoThread()
            self.video_thread.change_pixmap_signal.connect(self.update_image)
            self.video_thread.start()
        else:
            if self.video_thread:
                self.video_thread.stop()
            self.live_Button.setStyleSheet(BTN_RED)

    def handle_record_button(self):
        if self.record_button.isChecked():
            self.record_button.setText("● Recording")
            self.record_button.setStyleSheet(BTN_GREEN)
            if self.video_thread:
                self.video_thread.start_recording()
        else:
            self.record_button.setText("Start Recording")
            self.record_button.setStyleSheet(BTN_RED)
            if self.video_thread:
                self.video_thread.stop_recording()

    def snap_DIA_image(self):
        self.image_path = "C:\\Users\\Cell Culture Scope\\Pictures\\image.png"
        self.mmc.setProperty(self.core, 'Shutter', self.DIAshutter)
        self.mmc.waitForSystem()
        self.mmc.setExposure(self.camera, 20)
        self.mmc.setProperty(self.camera, 'CONVERSION FACTOR COEFF', 0.5)
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        self.mmc.waitForDevice(self.filter)
        self.mmc.waitForSystem()
        self.mmc.setAutoShutter(True)
        self.mmc.snapImage()
        rawImage    = self.mmc.getImage()
        image_width = self.mmc.getImageWidth()
        image_height = self.mmc.getImageHeight()

        rawImage = np.frombuffer(rawImage, dtype=np.uint16).reshape((image_height, image_width)).T
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        raw_data = self.adjusted_image.tobytes()
        self.myQImage = QImage(raw_data, image_width, image_height, QImage.Format_Grayscale16)
        self.image_Live.setPixmap(QPixmap.fromImage(self.myQImage).scaled(
            self.image_Live.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))

    def snap_EPI_image(self, Filter, Exposure_value):
        self.image_path = "C:\\Users\\Cell Culture Scope\\Pictures\\image.png"
        self.mmc.setProperty(self.core, 'Shutter', self.EPIshutter)
        self.mmc.waitForSystem()
        self.change_filter(Filter)
        self.mmc.setExposure(self.camera, Exposure_value)
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        self.mmc.waitForDevice(self.filter)
        self.mmc.waitForSystem()
        self.mmc.snapImage()
        rawImage = self.mmc.getImage()
        image_width  = self.mmc.getImageWidth()
        image_height = self.mmc.getImageHeight()

        rawImage = np.frombuffer(rawImage, dtype=np.uint16).reshape((image_height, image_width)).T
        scaled = np.clip((rawImage - 1500) / (14500 - 1500), 0, 1)
        scaled = (scaled * 65535).astype(np.uint16)
        self.adjusted_image = scaled
        raw_data_rescaled = self.adjusted_image.tobytes()
        self.myQImage = QImage(raw_data_rescaled, image_width, image_height, QImage.Format_Grayscale16)
        self.myQImage.save('C:\\Users\\Cell Culture Scope\\Pictures\\image_Qimage.png')
        tiff.imwrite('C:\\Users\\Cell Culture Scope\\Pictures\\rawimage.png', rawImage)
        self.image_Live.setPixmap(QPixmap.fromImage(self.myQImage).scaled(
            self.image_Live.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        return self.myQImage

    def save_image(self, save_path=None):
        if self.image_Live.pixmap():
            self.image_Live.pixmap().save(self.image_path)
            print(f"Image saved to {self.image_path}")
        else:
            print("No image to save")



    @pyqtSlot(np.ndarray)
    def update_image(self, cv_img):
        h, w = cv_img.shape
        raw_data = cv_img.tobytes()
        qimg = QImage(raw_data, w, h, QImage.Format_Grayscale16)
        self.image_Live.setPixmap(QPixmap.fromImage(qimg).scaled(
            self.image_Live.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))

    def DIAlamp_ON(self):
        if self.dialamponlight.isChecked():
            self.mmc.setProperty(self.DIAlamp, 'State', 1)
            self.dialamponlight.setStyleSheet(BTN_GREEN)
        else:
            self.mmc.setProperty(self.DIAlamp, 'State', 0)
            self.dialamponlight.setStyleSheet(BTN_RED)

    def save_Position(self):
        if len(self.positions) < 8:
            self.positions.append(list(self.get_new_position()))
            self.updateTable()
        else:
            QMessageBox.warning(self, 'Limit Reached', 'Cannot save more than 8 positions.')

    def get_new_position(self):
        return (self.mmc.getXPosition(self.stage),
                self.mmc.getYPosition(self.stage),
                self.mmc.getPosition())

    def updateTable(self):
        for row, (x, y, z) in enumerate(self.positions):
            self.Positions_table.setItem(row, 0, QTableWidgetItem(f'{x:.2f}'))
            self.Positions_table.setItem(row, 1, QTableWidgetItem(f'{y:.2f}'))
            self.Positions_table.setItem(row, 2, QTableWidgetItem(f'{z:.2f}'))
        for row in range(len(self.positions), 8):
            for col in range(3):
                self.Positions_table.setItem(row, col, QTableWidgetItem(''))

    def clearPositions(self):
        self.positions = []
        self.updateTable()

    def replacePosition(self, position_number):
        if not self.positions:
            QMessageBox.warning(self, 'No Positions', 'No positions available to replace.')
            return
        
        idx = position_number - 1
        if idx < len(self.positions):
            self.positions[idx] = list(self.get_new_position())
            self.updateTable()
    
    def GoToPosition(self, position_number):
        
        idx = position_number - 1
        if idx < len(self.positions):
            self.mmc.setXYPosition(self.positions[idx][0], self.positions[idx][1])
            self.mmc.setPosition(self.positions[idx][2])
            self.mmc.waitForSystem(); time.sleep(0.5)

    def PathtoCamera(self):
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)

    def Set_zoom(self, item):
        self.mmc.setProperty(self.zoom, 'Label', item)

    def change_filter(self, filter_name):
        self.mmc.setProperty(self.filter, 'State', filter_name - 1)

    def Set_stage_speed(self, item):
        self.stage_speed = int(item)

    def toggle_DIA_shutter(self):
        if self.DIAshutterbutton.isChecked():
            self.mmc.setProperty(self.DIAshutter, 'State', 1)
            self.DIAshutterbutton.setStyleSheet(BTN_GREEN)
        else:
            self.mmc.setProperty(self.DIAshutter, 'State', 0)
            self.DIAshutterbutton.setStyleSheet(BTN_RED)

    def toggle_EPI_shutter(self):
        if self.EPIshutterbutton.isChecked():
            self.mmc.setProperty(self.EPIshutter, 'State', 1)
            self.EPIshutterbutton.setStyleSheet(BTN_GREEN)
        else:
            self.mmc.setProperty(self.EPIshutter, 'State', 0)
            self.EPIshutterbutton.setStyleSheet(BTN_RED)

    # ── valve slots ───────────────────────────
    def get_relay_id(self, idx):
        return str(idx) if idx <= 9 else chr(ord('A') + (idx - 10))

    def control_valve(self, idx, state):
        relay_id = self.get_relay_id(idx)
        if state:
            self.controls[idx].setStyleSheet(BTN_GREEN)
            self.send_relay_command(f"relay off {relay_id}")
        else:
            self.controls[idx].setStyleSheet(BTN_RED)
            self.send_relay_command(f"relay on {relay_id}")

    def stop_all_callback(self):
        for i, ctrl in enumerate(self.controls):
            ctrl.setStyleSheet(BTN_RED); ctrl.setChecked(False)
            self.send_relay_command(f"relay on {self.get_relay_id(i)}")
        self.send_relay_command('open all')

    def all_on_callback(self):
        for i, ctrl in enumerate(self.controls):
            ctrl.setStyleSheet(BTN_GREEN); ctrl.setChecked(True)
            self.send_relay_command(f"relay off {self.get_relay_id(i)}")
        self.send_relay_command('close all')

    def send_relay_command(self, command):
        if self.Numato_port and self.Numato_port.is_open:
            try:
                self.Numato_port.write(f"{command}\r".encode('utf-8'))
                time.sleep(0.005)
            except serial.SerialException:
                QMessageBox.critical(self, 'Error', 'Failed to communicate with the device')

    # ── droplet slots ─────────────────────────
    def Characterize_Droplet(self, input, chemostat_number=1,
                              purge_duration=0, flow_duration=0, drive_duration=0):
        self.Characterize_Droplet_thread = DropletWorker(
            "characterize", input,
            purge_duration   = float(self.purge_duration_Input.text()),
            flow_duration    = float(self.flow_duration_Input.text()),
            drive_duration   = float(self.drive_duration_Input.text()),
            chemostat_number = int(self.inlet_Input.text()))
        self.Characterize_Droplet_thread.start()

    def PWM_droplet(self, input_1=12, input_2=13):
        self.PWM_thread = DropletWorker(
            "PWM", input_1, input_2,
            PWM_duration1    = float(self.PWM_duration1_Input.text()),
            PWM_duration2    = float(self.PWM_duration2_Input.text()),
            PWM_totalduration = float(self.PWM_totalduration_Input.text()))
        self.PWM_thread.start()

    def alter_Arduino_state(self, checked):
        if checked:
            self.TurnOnVolts.setText('Volts ON')
            self.TurnOnVolts.setStyleSheet(BTN_GREEN)
            send_to_arduino(2)
        else:
            self.TurnOnVolts.setText('Voltage Signal')
            self.TurnOnVolts.setStyleSheet(BTN_RED)
            send_to_arduino(3)

    # ── timelapse slots ───────────────────────
    def read_Exposure_values(self):
        self.selected_exposures = []
        for row in range(self.Exposures_table.rowCount()):
            filter = self.Exposures_table.cellWidget(row, 0).value()
            exposure = self.Exposures_table.cellWidget(row, 1).value()
            if exposure > 0:
                self.selected_exposures.append([filter, exposure])
        self.interval = int(self.cycle_Interval_Input.text())
        self.cycles   = int(self.cycle_Input.text())
        print("Exposures:", self.selected_exposures, "| Interval:", self.interval, "| Cycles:", self.cycles)

    def create_centered_checkbox(self):
        frame  = QFrame()
        layout = QHBoxLayout(frame)
        layout.setContentsMargins(0, 0, 0, 0)
        layout.setAlignment(Qt.AlignCenter)
        cb = QCheckBox(frame); cb.setEnabled(False)
        layout.addWidget(cb)
        return frame

    def add_loading_step(self):
        selected_inputs = [sb.value() for sb in self.Chemostat_inputs if sb.value() > 0]
        ring_status     = [cb.isChecked() for cb in self.Chemostats]
        if not selected_inputs and not any(ring_status):
            return
        step = {
            "input1": selected_inputs[0] if len(selected_inputs) > 0 else "",
            "input2": selected_inputs[1] if len(selected_inputs) > 1 else "",
            "rings":  ring_status
        }
        self.Chemostat_protocol_steps.append(step)
        col = self.current_protocol_table_step
        i1 = QTableWidgetItem(str(step["input1"])); i1.setTextAlignment(Qt.AlignCenter)
        i2 = QTableWidgetItem(str(step["input2"])); i2.setTextAlignment(Qt.AlignCenter)
        self.Chemostat_protocol_table.setItem(0, col, i1)
        self.Chemostat_protocol_table.setItem(1, col, i2)
        for i, is_on in enumerate(step["rings"]):
            cb = QCheckBox(); cb.setChecked(is_on); cb.setEnabled(False)
            w  = QWidget(); lyt = QVBoxLayout(w)
            lyt.setContentsMargins(0, 0, 0, 0)
            lyt.setAlignment(cb, Qt.AlignCenter)
            lyt.addWidget(cb)
            self.Chemostat_protocol_table.setCellWidget(2 + i, col, w)
        self.current_protocol_table_step += 1

    def clear_all_steps(self):
        self.Chemostat_protocol_steps = []
        self.current_protocol_table_step = 0
        for c in range(self.Chemostat_protocol_table.columnCount()):
            for r in range(self.Chemostat_protocol_table.rowCount()):
                if r < 2:
                    self.Chemostat_protocol_table.setItem(r, c, QTableWidgetItem(""))
                else:
                    self.Chemostat_protocol_table.setCellWidget(r, c, self.create_centered_checkbox())

    def clear_last_step(self):
        if not self.Chemostat_protocol_steps:
            return
        self.Chemostat_protocol_steps.pop()
        self.current_protocol_table_step = max(0, self.current_protocol_table_step - 1)
        col = self.current_protocol_table_step
        for r in range(self.Chemostat_protocol_table.rowCount()):
            if r < 2:
                self.Chemostat_protocol_table.setItem(r, col, QTableWidgetItem(""))
            else:
                frame = self.Chemostat_protocol_table.cellWidget(r, col)
                if frame:
                    cb = frame.layout().itemAt(0).widget()
                    if cb: cb.setChecked(False)

    def export_to_csv(self):
        if not self.Chemostat_protocol_steps:
            return
        fp, _ = QFileDialog.getSaveFileName(self, "Save Sequence to CSV", "",
                                             "CSV Files (*.csv);;All Files (*)")
        if not fp:
            return
        with open(fp, mode="w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["Step","Input 1","Input 2"] + [f"Ring {i+1}" for i in range(8)])
            for idx, step in enumerate(self.Chemostat_protocol_steps, 1):
                w.writerow([f"Step {idx}", step["input1"], step["input2"]]
                           + ["ON" if s else "OFF" for s in step["rings"]])

    def browse_folder(self):
        folder = QFileDialog.getExistingDirectory(self, "Select Folder")
        if folder:
            os.chdir(folder)
            self.directory_input.setText(folder)

    # ── timelapse experiment ──────────────────
    def TimeLapse_Experiment(self, num_loops, time_interval, positions_table,
                              selected_exposures, chemostat_protocol_table):
        init_states = [(0,False),(1,True),(2,False),(3,True),(4,True),(5,False),
                       (6,False),(7,True),(8,False),(9,False),(10,False),(11,False),
                       (12,True),(13,True),(14,True),(15,True)]
        for v, s in init_states:
            self.control_valve(v, state=s)

        purge_duration = float(self.purge_duration_Input.text())
        flow_duration  = float(self.flow_duration_Input.text())
        drive_duration = float(self.drive_duration_Input.text())

        for loop in range(num_loops):
            print(f"--- Loop {loop+1}/{num_loops} ---")
            loop_start = time.time()

            for ci in range(len(positions_table)):
                self.mmc.setXYPosition(positions_table[ci][0], positions_table[ci][1])
                self.mmc.setPosition(positions_table[ci][2])
                self.mmc.waitForSystem(); time.sleep(0.5)
                for filt, exp in selected_exposures:
                    img = self.snap_EPI_image(filt, exp)
                    fn  = f"Expt_{ci+1}_{filt}_{exp}_{loop+1}.tiff"
                    time.sleep(0.2)
                    img.save(os.path.join(".", fn))

            if self.Chemostat_protocol_steps:
                self.control_valve(15, state=False); time.sleep(30)
                self.control_valve(15, state=True)
                self.control_valve(7,  state=False)
                for v in [12, 13, 14, 15]:
                    self.control_valve(v, state=False); time.sleep(5)
                    self.control_valve(v, state=True)
                self.control_valve(7, state=True)
                self.mmc.setProperty(self.DIAlamp, 'State', 1)
                # FIX: stop any leftover thread from a previous loop iteration
                # before creating a new one, preventing camera resource conflicts
                if self.video_thread is not None and self.video_thread.isRunning():
                    self.video_thread.stop()
                    self.video_thread = None

                self.video_thread = VideoThread()
                self.video_thread.start()
                # Give the acquisition pipeline a moment to initialise before
                # the first start_recording() call
                QtCore.QThread.msleep(300)

                for step in self.Chemostat_protocol_steps:
                    i1 = step["input1"]; i2 = step["input2"]
                    pw = DropletWorker("purge", i1, i2, purge_duration=purge_duration)
                    pw.start(); pw.wait()
                    # ── Build active-connections map for this loop ─────────
                    # connections_map: {source_chemostat_1idx: target_chemostat_1idx}
                    # A connection fires on loop numbers that are multiples of
                    # connection_every_n (1-indexed loop counter).
                    connections_map = {}
                    every_n = self.connection_every_n.value()
                    if (loop + 1) % every_n == 0:
                        for (cb, sb_a, sb_b) in self.connection_rows:
                            if cb.isChecked():
                                src_ch = sb_a.value()   # 1-indexed chemostat
                                tgt_ch = sb_b.value()
                                if src_ch != tgt_ch:
                                    connections_map[src_ch] = tgt_ch
                    # Chemostats that are the *target* of a connection are
                    # skipped for their own drive step (source drives for both).
                    skipped_chemostats = set(connections_map.values())

                    for rn, active in enumerate(step["rings"]):
                        chemostat_1idx = rn + 1   # 1-indexed

                        # Skip this chemostat if it is the target of an
                        # active connection — the source will drive it.
                        if chemostat_1idx in skipped_chemostats:
                            print(f"  Chemostat {chemostat_1idx}: skipped (driven by connection)")
                            continue

                        if active:
                            self.mmc.setXYPosition(positions_table[rn][0], positions_table[rn][1])
                            self.mmc.setPosition(positions_table[rn][2])
                            gw = DropletWorker("generate", i1, flow_duration=flow_duration)
                            gw.start(); gw.wait()
                            # FIX: guard against a dead video thread before recording
                            if self.video_thread and self.video_thread.isRunning():
                                self.video_thread.start_recording()
                            else:
                                print(f"WARNING: VideoThread not running at loop {loop+1}, "
                                      f"step {rn+1} – skipping recording for this chemostat")
                                continue

                            # Resolve connection for this chemostat
                            do_connect  = chemostat_1idx in connections_map
                            target_ch   = connections_map.get(chemostat_1idx)   # None if not connecting

                            dw = DropletWorker(
                                "drive", i1,
                                drive_duration=drive_duration,
                                chemostat_number=chemostat_1idx,
                                connect=do_connect,
                                chemostat_number2=target_ch)
                            dw.start(); dw.wait()
                            # FIX: always stop recording in a finally block so a
                            # DropletWorker failure can't leave the writer open
                            if self.video_thread and self.video_thread.isRunning():
                                self.video_thread.stop_recording()
                            # Brief pause to let the writer flush before the next clip
                            QtCore.QThread.msleep(200)
                    self.control_valve(15, state=False); self.control_valve(7, state=False)
                    time.sleep(5)
                    self.control_valve(15, state=True);  self.control_valve(7, state=True)

                self.video_thread.stop()
                self.video_thread = None
                self.mmc.setProperty(self.DIAlamp, 'State', 0)

            elapsed   = time.time() - loop_start
            remaining = time_interval * 60 - elapsed
            if remaining > 0:
                print(f"Waiting {remaining:.1f}s …")
                time.sleep(remaining)

        print("--- Experiment complete ---")
        self.control_valve(0, state=True)

    def resizeEvent(self, event):
        """Lock the window to a fixed aspect ratio (width:height ≈ 1.16:1)."""
        new_w = event.size().width()
        new_h = int(round(new_w * self._aspect_h / self._aspect_w))
        if event.size().height() != new_h:
            self.resize(new_w, new_h)
        super().resizeEvent(event)

    def closeEvent(self, event):
        self.all_on_callback()
        if self.Numato_port and self.Numato_port.is_open:
            self.Numato_port.close()
        arduino.close()
        event.accept()


# ─────────────────────────────────────────────
if __name__ == '__main__':
    app = QtWidgets.QApplication(sys.argv)
    app.setStyle('Fusion')
    app.setPalette(make_dark_palette())
    app.setStyleSheet(WIDGET_STYLE)
    window = MicroscopeControlGUI()
    sys.exit(app.exec_())
